## About This Project

This notebook was built to demonstrate a production-style sales forecasting pipeline
using MLForecast and LightGBM — the same methodology used in revenue and demand
forecasting at mid-to-large companies.

The dataset is synthetic but mirrors a realistic multi-channel structure (Direct, Retail,
Online) with a growth trend and low seasonality — consistent with many B2B, fintech,
and subscription-based businesses where trend and momentum carry most of the signal.

To use with real data: replace the synthetic data in Section 2 with an exported CSV of
your own monthly sales, formatted with three columns — unique_id (channel or segment name),
ds (date), and y (sales value). No other code changes are required.

Built by Vincent Lepore Jr | Analytics & Data Science Portfolio  
Tools: MLForecast, LightGBM, scikit-learn, Plotly, pandas


# Sales Channel Forecasting with MLForecast + LightGBM

This notebook builds a multi-channel sales forecasting model using [MLForecast](https://github.com/Nixtla/mlforecast) from Nixtla.  
The approach converts time series data into a supervised regression problem using lag features, then trains a LightGBM model to predict future sales.

**Channels modeled:** Direct, Retail, Online  
**Model:** LightGBM via MLForecast  
**Forecast horizon:** 6 months  

---

## 1. Install & Import Dependencies

In [1]:
# Install required packages (run once in Colab or fresh env)
!pip install mlforecast lightgbm plotly pandas numpy --quiet

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings('ignore')

print('All imports successful.')

All imports successful.


## 2. Generate Synthetic Sales Data

Simulating 3 years of monthly sales across 3 channels.  
Data includes a growth trend and mild noise — no heavy seasonality, consistent with many mid-size B2B and fintech environments.

In [3]:
np.random.seed(42)

# Date range: 36 months of history
dates = pd.date_range(start='2021-01-01', periods=36, freq='MS')

channels = {
    'Direct':  500 + np.arange(36) * 8  + np.random.normal(0, 30, 36),
    'Retail':  300 + np.arange(36) * 5  + np.random.normal(0, 20, 36),
    'Online':  150 + np.arange(36) * 12 + np.random.normal(0, 25, 36),
}

# MLForecast expects long format with columns: unique_id, ds, y
dfs = []
for channel, values in channels.items():
    df_ch = pd.DataFrame({
        'unique_id': channel,
        'ds': dates,
        'y': values.round(2)
    })
    dfs.append(df_ch)

df = pd.concat(dfs, ignore_index=True)

print(f'Dataset shape: {df.shape}')
print(f'Channels: {df.unique_id.unique()}')
df.head(10)

Dataset shape: (108, 3)
Channels: ['Direct' 'Retail' 'Online']


,unique_id,ds,y
0,Direct,2021-01-01,514.90
1,Direct,2021-02-01,503.85
2,Direct,2021-03-01,535.43
3,Direct,2021-04-01,569.69
4,Direct,2021-05-01,524.98
5,Direct,2021-06-01,532.98
6,Direct,2021-07-01,595.38
7,Direct,2021-08-01,579.02
8,Direct,2021-09-01,549.92
9,Direct,2021-10-01,588.28


## 3. Visualize Historical Sales by Channel

In [4]:
fig = px.line(
    df,
    x='ds',
    y='y',
    color='unique_id',
    title='Historical Monthly Sales by Channel',
    labels={'ds': 'Month', 'y': 'Sales ($K)', 'unique_id': 'Channel'},
    template='plotly_white'
)
fig.update_layout(legend_title_text='Channel', hovermode='x unified')
fig.show()

## 4. Build the MLForecast Model

**How MLForecast works under the hood:**  
Rather than modeling the time series directly, MLForecast creates **lag features** from your target variable — e.g., the value from 1, 2, and 3 months ago — and treats predicting the next value as a standard regression problem.  
This lets you plug in any regression-compatible model (LightGBM, LinearRegression, XGBoost, etc.).

**Why LightGBM?**  
It handles non-linear relationships between lags well, is fast on tabular data, and is one of the most widely deployed models in production forecasting pipelines at mid-to-large companies.

In [5]:
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression

fcst = MLForecast(
    models={
        'LightGBM': LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1),
        'LinearRegression': LinearRegression()
    },
    freq='MS',
    lags=[1, 2, 3, 6],
    lag_transforms={
        1: [RollingMean(window_size=3)],
    },
    date_features=['month'],
)

print('Model configuration set.')
print('Lags used as features: [1, 2, 3, 6] months')
print('Models: LightGBM, LinearRegression (for comparison)')

Model configuration set.
Lags used as features: [1, 2, 3, 6] months
Models: LightGBM, LinearRegression (for comparison)


## 5. Train the Model

In [6]:
# Fit on full historical dataset
fcst.fit(df)

print('Model trained successfully on all 3 channels.')

Model trained successfully on all 3 channels.


## 6. Generate 6-Month Forecast

In [7]:
horizon = 6  # Forecast 6 months ahead

forecast_df = fcst.predict(horizon)
forecast_df.head(10)

,unique_id,ds,LightGBM,LinearRegression
0,Direct,2024-01-01,686.210327,762.501147
1,Direct,2024-02-01,686.210327,785.095367
2,Direct,2024-03-01,686.210327,777.385136
3,Direct,2024-04-01,686.210327,785.504873
4,Direct,2024-05-01,704.341585,797.083514
5,Direct,2024-06-01,704.341585,799.439441
6,Online,2024-01-01,600.473939,587.259736
7,Online,2024-02-01,641.908776,604.427217
8,Online,2024-03-01,627.074340,603.286022
9,Online,2024-04-01,677.111115,611.742147


## 7. Visualize Forecast vs. History

Each channel plotted separately with historical actuals and both model forecasts.

In [8]:
channel_list = df['unique_id'].unique()
colors = {'LightGBM': '#2196F3', 'LinearRegression': '#FF9800'}

fig = make_subplots(
    rows=len(channel_list), cols=1,
    subplot_titles=[f'{ch} Channel' for ch in channel_list],
    shared_xaxes=False,
    vertical_spacing=0.08
)

for i, channel in enumerate(channel_list, start=1):
    hist = df[df['unique_id'] == channel]
    fcast = forecast_df[forecast_df['unique_id'] == channel]

    # Historical actuals
    fig.add_trace(
        go.Scatter(x=hist['ds'], y=hist['y'], mode='lines',
                   name='Actuals' if i == 1 else '',
                   line=dict(color='#444', width=2),
                   showlegend=(i == 1)),
        row=i, col=1
    )

    # Forecast lines per model
    for model, color in colors.items():
        fig.add_trace(
            go.Scatter(x=fcast['ds'], y=fcast[model], mode='lines+markers',
                       name=model if i == 1 else '',
                       line=dict(color=color, width=2, dash='dash'),
                       showlegend=(i == 1)),
            row=i, col=1
        )

fig.update_layout(
    height=900,
    title_text='6-Month Sales Forecast by Channel — LightGBM vs Linear Regression',
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

## 8. Cross-Validation to Evaluate Model Accuracy

We use **time series cross-validation** — training on earlier windows and testing on later windows — to get a realistic error estimate.  
This avoids data leakage (a common mistake when evaluating forecasting models).

In [9]:
from mlforecast.utils import PredictionIntervals

# Cross-validate: 3 windows, each testing 6-month horizon
cv_df = fcst.cross_validation(
    df=df,
    h=6,           # Forecast horizon per window
    n_windows=3,   # Number of CV folds
)

cv_df.head(10)

,unique_id,ds,cutoff,y,LightGBM,LinearRegression
0,Direct,2022-07-01,2022-06-01,616.76,407.269724,634.212193
1,Direct,2022-08-01,2022-06-01,609.63,407.269724,624.848922
2,Direct,2022-09-01,2022-06-01,703.97,407.269724,640.755641
3,Direct,2022-10-01,2022-06-01,661.23,407.269724,648.137192
4,Direct,2022-11-01,2022-06-01,678.03,407.269724,648.966348
5,Direct,2022-12-01,2022-06-01,641.26,407.269724,661.362654
6,Online,2022-07-01,2022-06-01,368.43,407.269724,364.437377
7,Online,2022-08-01,2022-06-01,402.22,407.269724,364.418110
8,Online,2022-09-01,2022-06-01,372.45,407.269724,384.431815
9,Online,2022-10-01,2022-06-01,393.81,407.269724,393.033707


In [10]:
# Calculate MAE and RMSE per channel per model

def mae(actual, predicted):
    return np.mean(np.abs(actual - predicted))

def rmse(actual, predicted):
    return np.sqrt(np.mean((actual - predicted) ** 2))

results = []
for channel in cv_df['unique_id'].unique():
    ch_df = cv_df[cv_df['unique_id'] == channel]
    for model in ['LightGBM', 'LinearRegression']:
        results.append({
            'Channel': channel,
            'Model': model,
            'MAE': round(mae(ch_df['y'], ch_df[model]), 2),
            'RMSE': round(rmse(ch_df['y'], ch_df[model]), 2),
        })

results_df = pd.DataFrame(results).sort_values(['Channel', 'MAE'])
print('Model Accuracy — Cross-Validation Results')
print('=' * 50)
results_df

Model Accuracy — Cross-Validation Results


,Channel,Model,MAE,RMSE
1,Direct,LinearRegression,28.71,34.44
0,Direct,LightGBM,156.56,171.12
3,Online,LinearRegression,30.76,37.63
2,Online,LightGBM,72.78,93.60
5,Retail,LinearRegression,18.59,22.44
4,Retail,LightGBM,71.09,96.85


In [11]:
# Visualize error comparison
fig_err = px.bar(
    results_df,
    x='Channel',
    y='MAE',
    color='Model',
    barmode='group',
    title='Mean Absolute Error by Channel and Model (Cross-Validation)',
    template='plotly_white',
    labels={'MAE': 'Mean Absolute Error ($K)'}
)
fig_err.show()

## 9. LightGBM Feature Importance

Shows which lag features drove the most predictive power.  
In a non-seasonal business, you'd expect recent lags (lag_1, lag_2) and rolling averages to dominate — trend is carrying most of the signal.

In [12]:
# Preprocess the data the same way MLForecast does internally
preprocessed = fcst.preprocess(df)
print(preprocessed.head())

# Separate features and target
feature_cols = [c for c in preprocessed.columns if c not in ['unique_id', 'ds', 'y']]
X = preprocessed[feature_cols].dropna()
y = preprocessed.loc[X.index, 'y']

# Fit a standalone LightGBM on the full preprocessed dataset
from lightgbm import LGBMRegressor
import plotly.express as px

lgbm_standalone = LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
lgbm_standalone.fit(X, y)

# Extract and plot feature importance
fi_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': lgbm_standalone.feature_importances_
}).sort_values('Importance', ascending=False)

fig_fi = px.bar(
    fi_df,
    x='Importance',
    y='Feature',
    orientation='h',
    title='LightGBM Feature Importance (Lag Features)',
    template='plotly_white'
)
fig_fi.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_fi.show()

   unique_id         ds       y    lag1    lag2    lag3    lag6  \
6     Direct 2021-07-01  595.38  532.98  524.98  569.69  514.90   
7     Direct 2021-08-01  579.02  595.38  532.98  524.98  503.85   
8     Direct 2021-09-01  549.92  579.02  595.38  532.98  535.43   
9     Direct 2021-10-01  588.28  549.92  579.02  595.38  569.69   
10    Direct 2021-11-01  566.10  588.28  549.92  579.02  524.98   

    rolling_mean_lag1_window_size3  month  
6                       542.550000      7  
7                       551.113333      8  
8                       569.126667      9  
9                       574.773333     10  
10                      572.406667     11  


## 10. Summary & Next Steps

**What this model does:**
- Trains a LightGBM regressor on lag-based features derived from 36 months of multi-channel sales history
- Forecasts 6 months forward per channel independently
- Validates using time series cross-validation (no data leakage)
- Compares against a Linear Regression baseline

**When to use LightGBM over ARIMA:**
- Multiple series with shared patterns (channels, SKUs, regions)
- Non-linear relationships between lags
- When you want to add external regressors (promotions, pricing, macro data) easily

**Potential enhancements:**
- Add external regressors (price, marketing spend) via `fcst.fit(df, X_df=...)`
- Tune LightGBM hyperparameters with Optuna
- Replace synthetic data with real exported sales data
- Add prediction intervals for uncertainty quantification
